# Export Real-fMRI Macro States

Load `loc_model_stage2/stage2_real_fmri_macro/model_scale1.pkl`, encode each subject from `loc_data_real_fmri/generated_data.npz` at scale 1, apply one macro-dynamics step, and export one `yt` CSV plus one `yt+1` CSV per subject under `result/real_fmri/macro_state`.

The helpers also accept a 2D single-series dataset, so the notebook remains usable for Lorenz-style data by overriding the configuration cell.


In [1]:
from __future__ import annotations

import re
import sys
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Union

import numpy as np
import pandas as pd
import torch
from IPython.display import display

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "src" / "models_macro.py").exists() and (candidate / "loc_model_stage2").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

from src.models_macro import Parellel_Renorm_Dynamic

torch.set_grad_enabled(False)


In [2]:
# Default real-fMRI configuration.
DEFAULT_RUN_NAME = "stage2_real_fmri_macro"
DEFAULT_DATA_PATH = Path("loc_data_real_fmri") / "generated_data.npz"
DEFAULT_MODEL_NAME = "model_scale1.pkl"
DEFAULT_SUMMARY_NAME = "summary_scale1.csv"
DEFAULT_OUTPUT_DIR = Path("result") / "real_fmri" / "macro_state"
DEFAULT_DEVICE = "cpu"


In [3]:
def find_project_root(start: Optional[Path] = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "models_macro.py").exists() and (candidate / "loc_model_stage2").exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root containing src/models_macro.py and loc_model_stage2.")


def parse_int_list(value) -> List[int]:
    text = "" if value is None else str(value).strip()
    if text == "" or text.lower() == "nan":
        return []
    return [int(part.strip()) for part in text.split(",") if part.strip()]


def safe_subject_id(value, fallback_index: int) -> str:
    text = str(value).strip() if value is not None else ""
    if not text:
        text = f"subject_{fallback_index + 1:02d}"
    text = re.sub(r"[^0-9A-Za-z_.-]+", "_", text)
    return text or f"subject_{fallback_index + 1:02d}"


def load_macro_state_data(data_path: Path) -> Tuple[np.ndarray, List[int], List[str]]:
    with np.load(data_path, allow_pickle=False) as archive:
        if "data" not in archive:
            raise ValueError(f"Missing 'data' array in: {data_path}")
        origin_data = np.asarray(archive["data"], dtype=np.float32)
        group = parse_int_list(",".join(str(int(value)) for value in archive["group"].reshape(-1))) if "group" in archive else []
        if "subject_ids" in archive:
            subject_ids = [safe_subject_id(value, idx) for idx, value in enumerate(archive["subject_ids"].astype(str))]
        else:
            subject_count = int(origin_data.shape[0]) if origin_data.ndim == 3 else 1
            subject_ids = [safe_subject_id(None, idx) for idx in range(subject_count)]

    if origin_data.ndim not in {2, 3}:
        raise ValueError(f"Expected data with shape [T, N] or [S, T, N], but got {origin_data.shape}")
    feature_dim = int(origin_data.shape[-1])
    if group and sum(group) != feature_dim:
        raise ValueError(f"Group sizes must sum to {feature_dim}, but got {group}")
    if origin_data.ndim == 3 and len(subject_ids) != origin_data.shape[0]:
        raise ValueError(f"subject_ids length {len(subject_ids)} does not match subject count {origin_data.shape[0]}")
    return origin_data, group, subject_ids


def load_state_dict(model_path: Path, device: torch.device):
    try:
        return torch.load(model_path, map_location=device, weights_only=True)
    except TypeError:
        return torch.load(model_path, map_location=device)


In [4]:
def infer_scale_dims(state_dict) -> List[int]:
    layers_by_scale: Dict[int, List[Tuple[int, torch.Tensor]]] = {}
    for name, tensor in state_dict.items():
        match = re.match(r"^dynamics_modules\.(\d+)\.(\d+)\.weight$", name)
        if match:
            layers_by_scale.setdefault(int(match.group(1)), []).append((int(match.group(2)), tensor))
    if not layers_by_scale:
        raise ValueError("The checkpoint does not contain macro dynamics modules.")
    return [int(max(layers_by_scale[idx], key=lambda item: item[0])[1].shape[0]) for idx in sorted(layers_by_scale)]


def count_weight_layers(state_dict, pattern: str) -> int:
    return sum(1 for name in state_dict if re.match(pattern, name))


def infer_checkpoint_config(state_dict, summary_row: Dict, group: List[int]) -> Dict:
    scale_dims = infer_scale_dims(state_dict)
    encoder_type = "invertible" if any(".group_flows." in name or ".flow.flow." in name for name in state_dict) else str(summary_row.get("encoder_type", "mlp"))

    encoder_weight = state_dict.get("group_transitions.0.encoders.0.0.weight")
    dynamics_weight = state_dict.get("dynamics_modules.0.0.weight")
    hidden_units1 = int(encoder_weight.shape[0]) if encoder_weight is not None else int(summary_row.get("hidden_units1", 100))
    hidden_units2 = int(dynamics_weight.shape[0]) if dynamics_weight is not None else int(summary_row.get("hidden_units2", 100))

    dynamics_num_layers = count_weight_layers(state_dict, r"^dynamics_modules\.0\.\d+\.weight$")
    if dynamics_num_layers == 0:
        dynamics_num_layers = int(summary_row.get("dynamics_num_layers", 4))

    flow_num_layers = count_weight_layers(state_dict, r"^group_transitions\.0\.encoders\.0\.\d+\.weight$")
    if flow_num_layers == 0:
        flow_num_layers = int(summary_row.get("flow_num_layers", 3))

    return {
        "logical_scale_id": int(summary_row.get("scale_id", 0)),
        "summary_scale_dims": parse_int_list(summary_row.get("scale_dims", "")),
        "checkpoint_scale_dims": scale_dims,
        "latent_size": int(scale_dims[-1]),
        "hidden_units1": hidden_units1,
        "hidden_units2": hidden_units2,
        "flow_num_layers": flow_num_layers,
        "dynamics_num_layers": dynamics_num_layers,
        "reduce_dims": parse_int_list(summary_row.get("reduce_dims", "")),
        "group": group or parse_int_list(summary_row.get("group", "")),
        "encoder_type": encoder_type,
    }


def build_stage2_macro_model(
    num_nodes: int,
    model_path: Path,
    summary_path: Path,
    group: List[int],
    device: Union[str, torch.device] = "cpu",
):
    device = torch.device(device)
    summary_row = pd.read_csv(summary_path).iloc[0].to_dict()
    state_dict = load_state_dict(model_path, device=device)
    config = infer_checkpoint_config(state_dict, summary_row=summary_row, group=group)

    model = Parellel_Renorm_Dynamic(
        sym_size=int(num_nodes),
        latent_size=int(config["latent_size"]),
        effect_size=int(num_nodes),
        cut_size=2,
        hidden_units1=int(config["hidden_units1"]),
        hidden_units2=int(config["hidden_units2"]),
        normalized_state=True,
        device=device,
        is_random=False,
        flow_num_layers=int(config["flow_num_layers"]),
        dynamics_num_layers=int(config["dynamics_num_layers"]),
        decode_noise_scale=0.0,
        reduce_dims=config["reduce_dims"],
        group=config["group"],
        encoder_type=config["encoder_type"],
    ).to(device)
    model.load_state_dict(state_dict)
    model.eval()
    return model, config


In [5]:
def trim_trailing_nan_rows(series: np.ndarray) -> np.ndarray:
    series = np.asarray(series, dtype=np.float32)
    finite_rows = np.all(np.isfinite(series), axis=1)
    if finite_rows.all():
        return series
    invalid_rows = np.flatnonzero(~finite_rows)
    if invalid_rows.size == 0:
        return series
    return series[: int(invalid_rows[0])]


def encode_and_predict_next_macro_state(
    model: Parellel_Renorm_Dynamic,
    series_data: np.ndarray,
    logical_scale_id: int,
) -> Tuple[np.ndarray, np.ndarray]:
    series_data = trim_trailing_nan_rows(series_data)
    device = next(model.parameters()).device
    source = torch.as_tensor(series_data, dtype=torch.float32, device=device)
    with torch.no_grad():
        yt = model.encoding1(source, logical_scale_id)[logical_scale_id]
        yt_next = model._apply_dynamics(yt, logical_scale_id, inverse=False)
    return yt.cpu().numpy().astype(np.float32), yt_next.cpu().numpy().astype(np.float32)


def iter_subject_series(origin_data: np.ndarray, subject_ids: List[str]):
    if origin_data.ndim == 2:
        yield subject_ids[0] if subject_ids else "subject_01", origin_data
        return
    for idx, subject_series in enumerate(origin_data):
        yield subject_ids[idx] if idx < len(subject_ids) else f"subject_{idx + 1:02d}", subject_series


def export_macro_states(
    project_root: Optional[Path] = None,
    run_name: str = DEFAULT_RUN_NAME,
    data_path: Path = DEFAULT_DATA_PATH,
    model_name: str = DEFAULT_MODEL_NAME,
    summary_name: str = DEFAULT_SUMMARY_NAME,
    output_dir: Path = DEFAULT_OUTPUT_DIR,
    device: Union[str, torch.device] = DEFAULT_DEVICE,
) -> Tuple[pd.DataFrame, Dict[str, Tuple[np.ndarray, np.ndarray]]]:
    project_root = find_project_root(project_root)
    data_path = project_root / data_path
    model_path = project_root / "loc_model_stage2" / run_name / model_name
    summary_path = project_root / "loc_result_stage2" / run_name / summary_name
    output_dir = project_root / output_dir

    origin_data, group, subject_ids = load_macro_state_data(data_path)
    model, config = build_stage2_macro_model(
        num_nodes=int(origin_data.shape[-1]),
        model_path=model_path,
        summary_path=summary_path,
        group=group,
        device=device,
    )
    scale_id = int(config["logical_scale_id"])
    if not 0 <= scale_id < len(config["checkpoint_scale_dims"]):
        raise ValueError(f"logical_scale_id={scale_id} is outside checkpoint_scale_dims={config['checkpoint_scale_dims']}")

    output_dir.mkdir(parents=True, exist_ok=True)
    exported: Dict[str, Tuple[np.ndarray, np.ndarray]] = {}
    summary_rows = []
    for subject_id, subject_series in iter_subject_series(origin_data, subject_ids):
        safe_id = safe_subject_id(subject_id, len(summary_rows))
        yt, yt_next = encode_and_predict_next_macro_state(model, subject_series, scale_id)
        yt_path = output_dir / f"{safe_id}_yt.csv"
        yt_next_path = output_dir / f"{safe_id}_yt+1.csv"
        pd.DataFrame(yt).to_csv(yt_path, index=False, header=False)
        pd.DataFrame(yt_next).to_csv(yt_next_path, index=False, header=False)
        exported[safe_id] = (yt, yt_next)
        summary_rows.append(
            {
                "subject_id": safe_id,
                "data_path": str(data_path.resolve()),
                "model_path": str(model_path.resolve()),
                "summary_path": str(summary_path.resolve()),
                "logical_scale_id": scale_id,
                "summary_scale_dims": str(config["summary_scale_dims"]),
                "checkpoint_scale_dims": str(config["checkpoint_scale_dims"]),
                "encoder_type": config["encoder_type"],
                "num_samples": int(yt.shape[0]),
                "macro_dim": int(yt.shape[1]),
                "yt_csv": str(yt_path.resolve()),
                "yt_next_csv": str(yt_next_path.resolve()),
            }
        )

    summary = pd.DataFrame(summary_rows)
    summary.to_csv(output_dir / "macro_state_summary.csv", index=False)
    return summary, exported


In [6]:
PROJECT_ROOT = find_project_root()
EXPORT_SUMMARY, EXPORTED_MACRO_STATES = export_macro_states(project_root=PROJECT_ROOT)
EXPORT_SUMMARY


,subject_id,data_path,model_path,summary_path,logical_scale_id,summary_scale_dims,checkpoint_scale_dims,encoder_type,num_samples,macro_dim,yt_csv,yt_next_csv
0,A00028185,E:\code\Infer-Effective-Connection\loc_data_re...,E:\code\Infer-Effective-Connection\loc_model_s...,E:\code\Infer-Effective-Connection\loc_result_...,0,"[7, 3, 1]","[7, 3, 1]",mlp,895,7,E:\code\Infer-Effective-Connection\result\real...,E:\code\Infer-Effective-Connection\result\real...
1,A00033747,E:\code\Infer-Effective-Connection\loc_data_re...,E:\code\Infer-Effective-Connection\loc_model_s...,E:\code\Infer-Effective-Connection\loc_result_...,0,"[7, 3, 1]","[7, 3, 1]",mlp,895,7,E:\code\Infer-Effective-Connection\result\real...,E:\code\Infer-Effective-Connection\result\real...
2,A00035072,E:\code\Infer-Effective-Connection\loc_data_re...,E:\code\Infer-Effective-Connection\loc_model_s...,E:\code\Infer-Effective-Connection\loc_result_...,0,"[7, 3, 1]","[7, 3, 1]",mlp,895,7,E:\code\Infer-Effective-Connection\result\real...,E:\code\Infer-Effective-Connection\result\real...
3,A00035827,E:\code\Infer-Effective-Connection\loc_data_re...,E:\code\Infer-Effective-Connection\loc_model_s...,E:\code\Infer-Effective-Connection\loc_result_...,0,"[7, 3, 1]","[7, 3, 1]",mlp,895,7,E:\code\Infer-Effective-Connection\result\real...,E:\code\Infer-Effective-Connection\result\real...
4,A00035840,E:\code\Infer-Effective-Connection\loc_data_re...,E:\code\Infer-Effective-Connection\loc_model_s...,E:\code\Infer-Effective-Connection\loc_result_...,0,"[7, 3, 1]","[7, 3, 1]",mlp,895,7,E:\code\Infer-Effective-Connection\result\real...,E:\code\Infer-Effective-Connection\result\real...
5,A00037112,E:\code\Infer-Effective-Connection\loc_data_re...,E:\code\Infer-Effective-Connection\loc_model_s...,E:\code\Infer-Effective-Connection\loc_result_...,0,"[7, 3, 1]","[7, 3, 1]",mlp,895,7,E:\code\Infer-Effective-Connection\result\real...,E:\code\Infer-Effective-Connection\result\real...
6,A00037511,E:\code\Infer-Effective-Connection\loc_data_re...,E:\code\Infer-Effective-Connection\loc_model_s...,E:\code\Infer-Effective-Connection\loc_result_...,0,"[7, 3, 1]","[7, 3, 1]",mlp,895,7,E:\code\Infer-Effective-Connection\result\real...,E:\code\Infer-Effective-Connection\result\real...
7,A00038998,E:\code\Infer-Effective-Connection\loc_data_re...,E:\code\Infer-Effective-Connection\loc_model_s...,E:\code\Infer-Effective-Connection\loc_result_...,0,"[7, 3, 1]","[7, 3, 1]",mlp,895,7,E:\code\Infer-Effective-Connection\result\real...,E:\code\Infer-Effective-Connection\result\real...
8,A00039391,E:\code\Infer-Effective-Connection\loc_data_re...,E:\code\Infer-Effective-Connection\loc_model_s...,E:\code\Infer-Effective-Connection\loc_result_...,0,"[7, 3, 1]","[7, 3, 1]",mlp,895,7,E:\code\Infer-Effective-Connection\result\real...,E:\code\Infer-Effective-Connection\result\real...
9,A00039431,E:\code\Infer-Effective-Connection\loc_data_re...,E:\code\Infer-Effective-Connection\loc_model_s...,E:\code\Infer-Effective-Connection\loc_result_...,0,"[7, 3, 1]","[7, 3, 1]",mlp,895,7,E:\code\Infer-Effective-Connection\result\real...,E:\code\Infer-Effective-Connection\result\real...


In [7]:
first_subject = EXPORT_SUMMARY.iloc[0]["subject_id"]
yt, yt_next = EXPORTED_MACRO_STATES[first_subject]
preview = pd.concat(
    {
        f"{first_subject}_yt": pd.DataFrame(yt).head(),
        f"{first_subject}_yt+1": pd.DataFrame(yt_next).head(),
    },
    axis=1,
)
display(preview)
print(f"subjects exported: {len(EXPORTED_MACRO_STATES)}")
print(f"first subject: {first_subject}")
print(f"yt shape: {yt.shape}")
print(f"yt+1 shape: {yt_next.shape}")
print(f"output_dir: {PROJECT_ROOT / DEFAULT_OUTPUT_DIR}")


A00028185_yt                                                              \
             0         1         2         3         4         5         6   
0    -0.228387 -0.017355  0.019990  0.072226  0.476343 -0.208165  0.044872   
1    -0.277164 -0.011655 -0.019010  0.085928  0.472154 -0.245572 -0.013371   
2    -0.330627 -0.005699 -0.065440  0.098208  0.458592 -0.282670 -0.070441   
3    -0.387276  0.000357 -0.115063  0.116737  0.443183 -0.318362 -0.135254   
4    -0.444832  0.004488 -0.171016  0.140977  0.423914 -0.350525 -0.202223   

  A00028185_yt+1                                                              
               0         1         2         3         4         5         6  
0      -0.238616 -0.040334  0.015048  0.082304  0.465046 -0.207575  0.037937  
1      -0.285645 -0.034696 -0.025013  0.098233  0.462422 -0.245442 -0.022941  
2      -0.335071 -0.027932 -0.072574  0.112867  0.451494 -0.282368 -0.082266  
3      -0.385624 -0.020941 -0.123212  0.134074  0.438902 -0.317600 -0.149045  
4      -0.434604 -0.015503 -0.179336  0.160765  0.422679 -0.348953 -0.216589

subjects exported: 10
first subject: A00028185
yt shape: (895, 7)
yt+1 shape: (895, 7)
output_dir: E:\code\Infer-Effective-Connection\result\real_fmri\macro_state
